In [ ]:
from pathlib import Path

PROJECT_DIR = Path(r"C:\Users\ASUS\SIC-Food")

NEW_DATASET_DIR = (
    PROJECT_DIR
    / "datasets"
    / "raw"
    / "state_sources"
    / "Fruit_Vegetable_Healthy_Rotten"
)

NEW_DATASET_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print("Thư mục lưu:", NEW_DATASET_DIR)
print("Tồn tại:", NEW_DATASET_DIR.exists())

In [ ]:
import kagglehub

dataset_path = kagglehub.dataset_download(
    "muhammad0subhan/fruit-and-vegetable-disease-healthy-vs-rotten",
    output_dir=str(NEW_DATASET_DIR)
)

print("TẢI DATASET THÀNH CÔNG")
print("Đường dẫn:", dataset_path)

In [ ]:
from collections import Counter
import pandas as pd

DATASET_ROOT = Path(dataset_path)

EXTENSIONS = {".jpg", ".jpeg", ".png", ".webp"}

records = []

for folder in DATASET_ROOT.rglob("*"):
    if not folder.is_dir():
        continue

    images = [
        p for p in folder.iterdir()
        if p.is_file()
        and p.suffix.lower() in EXTENSIONS
    ]

    if images:
        records.append({
            "class": folder.name,
            "images": len(images),
            "path": str(folder)
        })

df_classes = pd.DataFrame(records)

display(df_classes.sort_values("class"))

print("Tổng file ảnh:", df_classes["images"].sum())

In [ ]:
TARGET_CLASSES = {
    "tomato": "tomato",
    "carrot": "carrot",
    "cucumber": "cucumber",
    "bellpepper": "bell_pepper",
    "potato": "potato",
    "mango": "mango",
    "strawberry": "strawberry"
}

def identify_class(folder_name):
    normalized = folder_name.lower().replace(
        "_", ""
    ).replace("-", "").replace(" ", "")

    ingredient = None

    for keyword, canonical in TARGET_CLASSES.items():
        if keyword in normalized:
            ingredient = canonical
            break

    if ingredient is None:
        return None, None

    if "healthy" in normalized:
        state = "normal_appearance"
    elif "rotten" in normalized:
        state = "visible_defect"
    else:
        state = None

    return ingredient, state

mapping_records = []

for row in df_classes.to_dict("records"):
    ingredient, state = identify_class(row["class"])

    if ingredient is None or state is None:
        continue

    mapping_records.append({
        "original_class": row["class"],
        "ingredient": ingredient,
        "state": state,
        "target_class": f"{ingredient}__{state}",
        "images": row["images"],
        "source_folder": row["path"]
    })

df_mapping = pd.DataFrame(mapping_records)

display(df_mapping)

print(
    "Tổng ảnh ứng viên cho 7 nguyên liệu:",
    df_mapping["images"].sum()
    if not df_mapping.empty else 0
)

In [ ]:
from PIL import Image
from IPython.display import display
import random

rng = random.Random(42)

for row in df_mapping.to_dict("records"):
    folder = Path(row["source_folder"])

    images = [
        p for p in folder.iterdir()
        if p.is_file()
        and p.suffix.lower() in EXTENSIONS
    ]

    print("\n" + "=" * 50)
    print("Lớp:", row["original_class"])
    print("Nhãn dự kiến:", row["target_class"])
    print("Số ảnh:", len(images))

    for image_path in rng.sample(
        images, min(2, len(images))
    ):
        with Image.open(image_path) as image:
            preview = image.convert("RGB")
            preview.thumbnail((250, 250))
            display(preview)

In [ ]:
from pathlib import Path
import pandas as pd

PROJECT_DIR = Path(r"C:\Users\ASUS\SIC-Food")
REPORT_DIR = PROJECT_DIR / "datasets" / "reports"
REPORT_DIR.mkdir(parents=True, exist_ok=True)

TARGET_INGREDIENTS = {
    "tomato", "carrot", "cucumber",
    "bell_pepper", "potato", "mango", "strawberry"
}

expected_states = {
    "normal_appearance",
    "visible_defect"
}

print("Các lớp tìm được:")
display(df_mapping[
    ["original_class", "target_class", "images"]
].sort_values("target_class"))

found = set(
    zip(df_mapping["ingredient"], df_mapping["state"])
)

expected = {
    (ingredient, state)
    for ingredient in TARGET_INGREDIENTS
    for state in expected_states
}

missing = expected - found

print("\nSố cặp nguyên liệu/trạng thái:", len(found))
print("Số cặp còn thiếu:", len(missing))
print("Các cặp còn thiếu:", sorted(missing))
print("Tổng ảnh ứng viên:", df_mapping["images"].sum())

df_mapping.to_csv(
    REPORT_DIR / "efficientnet_v2_label_mapping.csv",
    index=False,
    encoding="utf-8-sig"
)

In [ ]:
from pathlib import Path
from PIL import Image
from tqdm.auto import tqdm
import hashlib
import pandas as pd

EXTENSIONS = {".jpg", ".jpeg", ".png", ".webp"}

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

records = []
errors = []
seen_paths = set()

for row in df_mapping.to_dict("records"):
    folder = Path(row["source_folder"])

    for path in tqdm(
        list(folder.iterdir()),
        desc=row["target_class"]
    ):
        if not path.is_file():
            continue

        if path.suffix.lower() not in EXTENSIONS:
            continue

        resolved = str(path.resolve()).lower()

        if resolved in seen_paths:
            continue

        seen_paths.add(resolved)

        try:
            with Image.open(path) as img:
                img.verify()

            records.append({
                "path": str(path),
                "ingredient": row["ingredient"],
                "state": row["state"],
                "target_class": row["target_class"],
                "original_class": row["original_class"],
                "sha256": sha256_file(path)
            })

        except Exception as e:
            errors.append({
                "path": str(path),
                "error": str(e)
            })

df_new = pd.DataFrame(records)

print("Tổng ảnh đọc được:", len(df_new))
print("Ảnh lỗi:", len(errors))
print("Số bản dư trùng SHA256:",
      df_new.duplicated("sha256").sum())

display(
    df_new.groupby("target_class")
    .size()
    .reset_index(name="valid_images")
)

df_new.to_csv(
    REPORT_DIR / "efficientnet_v2_raw_manifest.csv",
    index=False,
    encoding="utf-8-sig"
)

pd.DataFrame(
    errors,
    columns=["path", "error"]
).to_csv(
    REPORT_DIR / "efficientnet_v2_errors.csv",
    index=False,
    encoding="utf-8-sig"
)

In [ ]:
conflicts = (
    df_new.groupby("sha256")["target_class"]
    .nunique()
)

conflict_hashes = set(
    conflicts[conflicts > 1].index
)

df_conflicts = df_new[
    df_new["sha256"].isin(conflict_hashes)
].sort_values("sha256")

print("Số nhóm ảnh trùng nhưng khác nhãn:",
      len(conflict_hashes))

display(df_conflicts.head(20))

df_conflicts.to_csv(
    REPORT_DIR / "efficientnet_v2_label_conflicts.csv",
    index=False,
    encoding="utf-8-sig"
)

In [ ]:
df_candidates = df_new[
    ~df_new["sha256"].isin(conflict_hashes)
].copy()

df_candidates = (
    df_candidates
    .drop_duplicates(subset="sha256", keep="first")
    .reset_index(drop=True)
)

print("Ảnh ứng viên sau lọc:", len(df_candidates))
print("Số lớp:", df_candidates["target_class"].nunique())

display(
    df_candidates.groupby("target_class")
    .size()
    .reset_index(name="images")
)

df_candidates.to_csv(
    REPORT_DIR / "efficientnet_v2_candidates.csv",
    index=False,
    encoding="utf-8-sig"
)

In [ ]:
%pip install imagehash scikit-learn

In [ ]:
from pathlib import Path
from PIL import Image
from tqdm.auto import tqdm
import imagehash
import pandas as pd

PROJECT_DIR = Path(r"C:\Users\ASUS\SIC-Food")
REPORT_DIR = PROJECT_DIR / "datasets" / "reports"
REPORT_DIR.mkdir(parents=True, exist_ok=True)

df_work = df_candidates.copy().reset_index(drop=True)

hashes = []
errors = []

for row in tqdm(
    df_work.itertuples(index=True),
    total=len(df_work),
    desc="Kiểm tra pHash"
):
    try:
        with Image.open(row.path) as img:
            h = imagehash.phash(img.convert("RGB"))
        hashes.append(str(h))
    except Exception as e:
        hashes.append(None)
        errors.append({
            "path": row.path,
            "error": str(e)
        })

df_work["phash"] = hashes

print("Số ảnh:", len(df_work))
print("Ảnh lỗi:", len(errors))

df_work.to_csv(
    REPORT_DIR / "v2_phash_manifest.csv",
    index=False,
    encoding="utf-8-sig"
)

In [ ]:
valid = df_work.dropna(subset=["phash"]).copy()

conflict_counts = (
    valid.groupby("phash")["target_class"]
    .nunique()
)

conflict_phashes = set(
    conflict_counts[conflict_counts > 1].index
)

phash_conflicts = valid[
    valid["phash"].isin(conflict_phashes)
]

print(
    "Số nhóm pHash giống nhau nhưng khác nhãn:",
    len(conflict_phashes)
)

display(phash_conflicts.head(20))

phash_conflicts.to_csv(
    REPORT_DIR / "v2_phash_conflicts.csv",
    index=False,
    encoding="utf-8-sig"
)

In [ ]:
import re
from collections import defaultdict

# Loại các ảnh có pHash mâu thuẫn nhãn
df_clean = valid[
    ~valid["phash"].isin(conflict_phashes)
].copy().reset_index(drop=True)

patterns = [
    r"^rotated_by_[+-]?\d+(?:\.\d+)?_",
    r"^vertical_flip_?",
    r"^horizontal_flip_?",
    r"^salt_and_pepper_?",
]

def original_filename(name):
    name = name.lower()

    changed = True
    while changed:
        changed = False

        for pattern in patterns:
            new_name = re.sub(pattern, "", name)
            if new_name != name:
                name = new_name
                changed = True

    return name

# Gom nhóm bằng union-find
parent = list(range(len(df_clean)))

def find(x):
    while parent[x] != x:
        parent[x] = parent[parent[x]]
        x = parent[x]
    return x

def union(a, b):
    ra, rb = find(a), find(b)
    if ra != rb:
        parent[rb] = ra

seen_hash = {}
seen_name = {}

for i, row in enumerate(df_clean.itertuples(index=False)):
    class_name = row.target_class
    hash_key = row.phash

    if hash_key in seen_hash:
        union(i, seen_hash[hash_key])
    else:
        seen_hash[hash_key] = i

    # Không gộp chỉ vì tên file phổ biến giống nhau
    # giữa các lớp khác nhau
    filename = original_filename(Path(row.path).name)
    name_key = (class_name, filename)

    if name_key in seen_name:
        union(i, seen_name[name_key])
    else:
        seen_name[name_key] = i

df_clean["group_id"] = [
    find(i) for i in range(len(df_clean))
]

group_conflicts = (
    df_clean.groupby("group_id")["target_class"].nunique()
)

bad_groups = set(
    group_conflicts[group_conflicts > 1].index
)

df_clean = df_clean[
    ~df_clean["group_id"].isin(bad_groups)
].copy()

print("Ảnh còn lại:", len(df_clean))
print("Số nhóm:", df_clean["group_id"].nunique())
print("Nhóm nhiều nhãn được cách ly:", len(bad_groups))

In [ ]:
import random
import pandas as pd

rng = random.Random(42)
group_split = {}

for class_name, part in df_clean.groupby("target_class"):

    group_ids = sorted(part["group_id"].unique())
    rng.shuffle(group_ids)

    n = len(group_ids)

    if n < 10:
        raise ValueError(
            f"{class_name} chỉ có {n} nhóm, "
            "chưa đủ để chia tập đáng tin cậy."
        )

    n_train = int(n * 0.70)
    n_val = int(n * 0.15)

    for g in group_ids[:n_train]:
        group_split[g] = "train"

    for g in group_ids[n_train:n_train+n_val]:
        group_split[g] = "val"

    for g in group_ids[n_train+n_val:]:
        group_split[g] = "test"

df_clean["split"] = df_clean["group_id"].map(group_split)

assert df_clean.groupby("group_id")["split"].nunique().max() == 1

summary = pd.crosstab(
    df_clean["target_class"],
    df_clean["split"]
)

display(summary)

print("Tổng ảnh:", len(df_clean))
print("Số lớp:", df_clean["target_class"].nunique())

df_clean.to_csv(
    REPORT_DIR / "efficientnet_v2_split_plan.csv",
    index=False,
    encoding="utf-8-sig"
)

In [ ]:
import shutil
import hashlib
from pathlib import Path
from tqdm.auto import tqdm
import pandas as pd

OUTPUT_DIR = PROJECT_DIR / "datasets" / "efficientnet_v2_7"

# Chặn ghi đè kết quả cũ
if OUTPUT_DIR.exists() and any(OUTPUT_DIR.rglob("*.*")):
    raise RuntimeError(
        "Thư mục đầu ra đã có file. "
        "Hãy kiểm tra trước khi xuất lại."
    )

export_records = []
errors = []

for row in tqdm(
    df_clean.to_dict("records"),
    desc="Export EfficientNet V2"
):
    source = Path(row["path"])

    target_dir = (
        OUTPUT_DIR
        / row["split"]
        / row["target_class"]
    )

    target_dir.mkdir(parents=True, exist_ok=True)

    uid = hashlib.sha256(
        str(source.resolve()).encode("utf-8")
    ).hexdigest()[:20]

    target = target_dir / (uid + source.suffix.lower())

    try:
        shutil.copy2(source, target)

        export_records.append({
            **row,
            "export_path": str(target)
        })

    except Exception as e:
        errors.append({
            "source": str(source),
            "error": str(e)
        })

df_export = pd.DataFrame(export_records)

df_export.to_csv(
    REPORT_DIR / "efficientnet_v2_export.csv",
    index=False,
    encoding="utf-8-sig"
)

pd.DataFrame(
    errors,
    columns=["source", "error"]
).to_csv(
    REPORT_DIR / "efficientnet_v2_export_errors.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Ảnh xuất thành công:", len(df_export))
print("Ảnh lỗi:", len(errors))
print("Đường dẫn:", OUTPUT_DIR)

In [ ]:
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

preprocess = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor()
])

train_ds = datasets.ImageFolder(
    OUTPUT_DIR / "train",
    transform=preprocess
)

val_ds = datasets.ImageFolder(
    OUTPUT_DIR / "val",
    transform=preprocess
)

test_ds = datasets.ImageFolder(
    OUTPUT_DIR / "test",
    transform=preprocess
)

assert train_ds.class_to_idx == val_ds.class_to_idx
assert train_ds.class_to_idx == test_ds.class_to_idx

print("Train:", len(train_ds))
print("Validation:", len(val_ds))
print("Test:", len(test_ds))
print("Số lớp:", len(train_ds.classes))

loader = DataLoader(
    train_ds,
    batch_size=8,
    num_workers=0
)

images, labels = next(iter(loader))

print("Batch shape:", images.shape)

In [ ]:
from pathlib import Path
from PIL import Image
from tqdm.auto import tqdm
import imagehash
import pandas as pd

PROJECT_DIR = Path(r"C:\Users\ASUS\SIC-Food")
DATASET_DIR = PROJECT_DIR / "datasets" / "efficientnet_v2_7"
REPORT_DIR = PROJECT_DIR / "datasets" / "reports"

REPORT_DIR.mkdir(parents=True, exist_ok=True)

extensions = {".jpg", ".jpeg", ".png", ".webp"}

records = []
errors = []

for split in ["train", "val", "test"]:
    for class_dir in sorted((DATASET_DIR / split).iterdir()):
        if not class_dir.is_dir():
            continue

        image_paths = [
            p for p in class_dir.iterdir()
            if p.is_file() and p.suffix.lower() in extensions
        ]

        for path in tqdm(image_paths, desc=f"{split}/{class_dir.name}"):
            try:
                with Image.open(path) as img:
                    phash = imagehash.phash(img.convert("RGB"))

                records.append({
                    "split": split,
                    "class": class_dir.name,
                    "path": str(path),
                    "phash": str(phash)
                })

            except Exception as e:
                errors.append({
                    "path": str(path),
                    "error": str(e)
                })

df_hash_v2 = pd.DataFrame(records)

df_hash_v2.to_csv(
    REPORT_DIR / "v2_7_phash_manifest.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Ảnh đã kiểm tra:", len(df_hash_v2))
print("Ảnh lỗi:", len(errors))

In [ ]:
import numpy as np
from scipy.spatial import cKDTree
from tqdm.auto import tqdm

def phash_to_bits(value):
    n = int(value, 16)
    return [(n >> i) & 1 for i in range(64)]

bits = np.asarray(
    [phash_to_bits(x) for x in df_hash_v2["phash"]],
    dtype=np.uint8
)

splits = df_hash_v2["split"].to_numpy()

pairs = []

def compare_splits(split_a, split_b, threshold=5):
    idx_a = np.where(splits == split_a)[0]
    idx_b = np.where(splits == split_b)[0]

    tree = cKDTree(bits[idx_b].astype(np.float32))

    for i in tqdm(idx_a, desc=f"{split_a} vs {split_b}"):
        neighbors = tree.query_ball_point(
            bits[i].astype(np.float32),
            r=np.sqrt(threshold) + 1e-6
        )

        for j_local in neighbors:
            j = idx_b[j_local]

            distance = int(
                np.count_nonzero(bits[i] != bits[j])
            )

            if distance <= threshold:
                pairs.append({
                    "image_a": df_hash_v2.iloc[i]["path"],
                    "split_a": split_a,
                    "class_a": df_hash_v2.iloc[i]["class"],
                    "image_b": df_hash_v2.iloc[j]["path"],
                    "split_b": split_b,
                    "class_b": df_hash_v2.iloc[j]["class"],
                    "distance": distance
                })

compare_splits("train", "val")
compare_splits("train", "test")
compare_splits("val", "test")

df_near_v2 = pd.DataFrame(
    pairs,
    columns=[
        "image_a", "split_a", "class_a",
        "image_b", "split_b", "class_b", "distance"
    ]
)

df_near_v2.to_csv(
    REPORT_DIR / "v2_7_near_duplicates.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Tổng cặp nghi gần trùng:", len(df_near_v2))

if not df_near_v2.empty:
    print("\nSố cặp theo từng nhóm split:")
    display(
        df_near_v2.groupby(["split_a", "split_b"])
        .size()
        .reset_index(name="pairs")
    )

    print("\nCác cặp nghi vấn:")
    display(df_near_v2.head(20))

In [ ]:
from PIL import Image
from IPython.display import display

if df_near_v2.empty:
    print("Không có cặp nghi gần trùng theo ngưỡng hiện tại.")
else:
    # Ưu tiên các cặp có pHash giống hệt
    samples = df_near_v2.sort_values("distance").head(10)

    for _, row in samples.iterrows():
        print("=" * 50)
        print("Khoảng cách pHash:", row["distance"])
        print("Ảnh A:", row["class_a"], row["split_a"])
        print("Ảnh B:", row["class_b"], row["split_b"])

        for key in ["image_a", "image_b"]:
            with Image.open(row[key]) as img:
                preview = img.convert("RGB")
                preview.thumbnail((260, 260))
                display(preview)

In [ ]:
import pandas as pd
from pathlib import Path

PROJECT_DIR = Path(r"C:\Users\ASUS\SIC-Food")
REPORT_DIR = PROJECT_DIR / "datasets" / "reports"

df_pairs = pd.read_csv(
    REPORT_DIR / "v2_7_near_duplicates.csv"
)

print("TỔNG SỐ CẶP:", len(df_pairs))

print("\n1. Phân bố khoảng cách pHash:")
display(
    df_pairs.groupby("distance")
    .size()
    .reset_index(name="pairs")
)

df_pairs["same_class"] = (
    df_pairs["class_a"] == df_pairs["class_b"]
)

print("\n2. So sánh nhãn:")
display(
    df_pairs.groupby("same_class")
    .size()
    .reset_index(name="pairs")
)

print("\n3. Số ảnh khác nhau bị đánh dấu:")
suspect_images = set(df_pairs["image_a"]) | set(df_pairs["image_b"])
print(len(suspect_images))

print("\n4. Số cặp theo từng lớp:")
display(
    df_pairs.groupby(["class_a", "distance"])
    .size()
    .unstack(fill_value=0)
)

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image
import pandas as pd

def show_duplicate_pairs(dataframe, start=0, count=8):

    subset = dataframe.iloc[start:start + count]

    if subset.empty:
        print("Không có ảnh ở vị trí này")
        return

    fig, axes = plt.subplots(
        len(subset), 2,
        figsize=(10, 3.5 * len(subset)),
        squeeze=False
    )

    for row_index, (_, row) in enumerate(subset.iterrows()):
        for col_index, key in enumerate(["image_a", "image_b"]):
            with Image.open(row[key]) as img:
                axes[row_index, col_index].imshow(
                    img.convert("RGB")
                )

            label_key = "class_a" if col_index == 0 else "class_b"
            split_key = "split_a" if col_index == 0 else "split_b"

            axes[row_index, col_index].set_title(
                f"{row[split_key]} | {row[label_key]}",
                fontsize=9
            )
            axes[row_index, col_index].axis("off")

        axes[row_index, 0].set_ylabel(
            f"Pair {subset.index[row_index]}\npHash={row['distance']}",
            fontsize=9
        )

    plt.tight_layout()
    plt.show()

# Ưu tiên xem các cặp pHash giống hệt
df_review = df_pairs.sort_values(
    ["distance", "class_a"]
).reset_index(drop=True)

show_duplicate_pairs(df_review, start=0, count=8)

In [ ]:
show_duplicate_pairs(df_review, start=8, count=8)

In [ ]:
show_duplicate_pairs(
    df_review[df_review["distance"] == 2].reset_index(drop=True),
    start=0,
    count=8
)

In [ ]:
review = df_review.copy()

review.insert(0, "pair_id", range(len(review)))

review["review_status"] = "pending"
review["review_note"] = ""

review_path = REPORT_DIR / "v2_7_duplicate_review.csv"

review.to_csv(
    review_path,
    index=False,
    encoding="utf-8-sig"
)

print("Đã tạo bảng kiểm duyệt:", review_path)

In [ ]:
from pathlib import Path
import pandas as pd

PROJECT_DIR = Path(r"C:\Users\ASUS\SIC-Food")
REPORT_DIR = PROJECT_DIR / "datasets" / "reports"

df_pairs = pd.read_csv(
    REPORT_DIR / "v2_7_near_duplicates.csv"
)

# Quy tắc ưu tiên: Train > Validation > Test
priority = {"train": 0, "val": 1, "test": 2}

quarantine = set()

for row in df_pairs.itertuples(index=False):
    if priority[row.split_a] < priority[row.split_b]:
        quarantine.add(str(Path(row.image_b).resolve()))
    else:
        quarantine.add(str(Path(row.image_a).resolve()))

print("Tổng cặp nghi trùng:", len(df_pairs))
print("Số ảnh riêng biệt cần cách ly:", len(quarantine))

quarantine_df = pd.DataFrame({
    "path": sorted(quarantine),
    "reason": "potential_cross_split_near_duplicate"
})

quarantine_df.to_csv(
    REPORT_DIR / "v2_7_quarantine.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Đã lưu danh sách cách ly.")

In [ ]:
manifest = pd.read_csv(
    REPORT_DIR / "v2_7_phash_manifest.csv"
)

manifest["path"] = manifest["path"].apply(
    lambda p: str(Path(p).resolve())
)

manifest["quarantined"] = manifest["path"].isin(quarantine)

df_safe = manifest[
    ~manifest["quarantined"]
].copy()

print("Ảnh ban đầu:", len(manifest))
print("Ảnh cách ly:", manifest["quarantined"].sum())
print("Ảnh còn lại:", len(df_safe))

summary = pd.crosstab(
    df_safe["class"],
    df_safe["split"]
)

display(summary)

required_splits = {"train", "val", "test"}

if not required_splits.issubset(summary.columns):
    raise RuntimeError(
        "Một hoặc nhiều split bị trống. Cần chia lại dữ liệu."
    )

if (summary[list(required_splits)] == 0).any().any():
    raise RuntimeError(
        "Có lớp không còn ảnh ở một split. "
        "Cần kiểm tra trước khi xuất."
    )

In [ ]:
import shutil
from tqdm.auto import tqdm

SOURCE_DIR = PROJECT_DIR / "datasets" / "efficientnet_v2_7"
CLEAN_DIR = PROJECT_DIR / "datasets" / "efficientnet_v2_7_clean"

# Không ghi đè dataset cũ
if CLEAN_DIR.exists() and any(CLEAN_DIR.rglob("*")):
    raise RuntimeError(
        "Thư mục clean đã có dữ liệu. "
        "Hãy kiểm tra trước khi chạy lại."
    )

exported = []
errors = []

for row in tqdm(
    df_safe.to_dict("records"),
    desc="Export clean dataset"
):
    src = Path(row["path"])

    relative = src.relative_to(SOURCE_DIR)
    dst = CLEAN_DIR / relative

    try:
        dst.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(src, dst)

        exported.append({
            **row,
            "clean_path": str(dst)
        })
    except Exception as e:
        errors.append({
            "path": str(src),
            "error": str(e)
        })

df_clean_export = pd.DataFrame(exported)

df_clean_export.to_csv(
    REPORT_DIR / "v2_7_clean_manifest.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Ảnh xuất thành công:", len(exported))
print("Ảnh lỗi:", len(errors))
print("Đường dẫn:", CLEAN_DIR)

In [ ]:
remaining = set(df_safe["path"])

remaining_pairs = df_pairs[
    df_pairs["image_a"].apply(
        lambda p: str(Path(p).resolve()) in remaining
    )
    &
    df_pairs["image_b"].apply(
        lambda p: str(Path(p).resolve()) in remaining
    )
]

print("Cặp nghi trùng còn lại:", len(remaining_pairs))

assert len(remaining_pairs) == 0, (
    "Vẫn còn cặp nghi trùng giữa các split"
)

print("\nSố ảnh theo split:")
print(
    df_clean_export.groupby("split").size()
)

print("\nSố lớp:", df_clean_export["class"].nunique())

In [ ]:
from pathlib import Path
import pandas as pd

PROJECT_DIR = Path(r"C:\Users\ASUS\SIC-Food")
REPORT_DIR = PROJECT_DIR / "datasets" / "reports"

OLD_DIR = PROJECT_DIR / "datasets" / "efficientnet"
OLD_CLEAN_DIR = PROJECT_DIR / "datasets" / "efficientnet_v2_3_clean"

near_path = REPORT_DIR / "near_duplicate_pairs.csv"

if not near_path.exists():
    raise FileNotFoundError(near_path)

df_old_pairs = pd.read_csv(near_path)

print("Số cặp nghi gần trùng:", len(df_old_pairs))

display(
    df_old_pairs.groupby(["split_a", "split_b"])
    .size()
    .reset_index(name="pairs")
)

In [ ]:
priority = {
    "train": 0,
    "val": 1,
    "test": 2
}

def normalize_path(path):
    return str(Path(path).resolve()).casefold()

quarantine_old = set()

for row in df_old_pairs.itertuples(index=False):

    if priority[row.split_a] < priority[row.split_b]:
        quarantine_old.add(normalize_path(row.image_b))
    else:
        quarantine_old.add(normalize_path(row.image_a))

print("Số ảnh cần cách ly:", len(quarantine_old))

pd.DataFrame({
    "path": sorted(quarantine_old)
}).to_csv(
    REPORT_DIR / "efficientnet_v2_3_quarantine.csv",
    index=False,
    encoding="utf-8-sig"
)

In [ ]:
import shutil
from tqdm.auto import tqdm

CLASSES_3 = [
    "apple__normal_appearance",
    "apple__visible_defect",
    "banana__normal_appearance",
    "banana__visible_defect",
    "orange__normal_appearance",
    "orange__visible_defect",
]

EXTENSIONS = {".jpg", ".jpeg", ".png", ".webp"}

if OLD_CLEAN_DIR.exists() and any(
    p.is_file() for p in OLD_CLEAN_DIR.rglob("*")
):
    raise RuntimeError(
        "Thư mục đầu ra đã có dữ liệu. "
        "Không ghi đè, hãy kiểm tra trước."
    )

records = []
errors = []

for split in ["train", "val", "test"]:
    for class_name in CLASSES_3:

        source_dir = OLD_DIR / split / class_name

        if not source_dir.is_dir():
            raise FileNotFoundError(source_dir)

        images = [
            p for p in source_dir.iterdir()
            if p.is_file()
            and p.suffix.lower() in EXTENSIONS
        ]

        for source in tqdm(
            images, desc=f"{split}/{class_name}"
        ):
            if normalize_path(source) in quarantine_old:
                continue

            target_dir = (
                OLD_CLEAN_DIR / split / class_name
            )

            target_dir.mkdir(
                parents=True,
                exist_ok=True
            )

            target = target_dir / source.name

            try:
                shutil.copy2(source, target)

                records.append({
                    "split": split,
                    "class": class_name,
                    "source": str(source),
                    "target": str(target)
                })
            except Exception as e:
                errors.append({
                    "source": str(source),
                    "error": str(e)
                })

df_old_clean = pd.DataFrame(records)

df_old_clean.to_csv(
    REPORT_DIR / "efficientnet_v2_3_clean_manifest.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Ảnh đã xuất:", len(df_old_clean))
print("Ảnh lỗi:", len(errors))

display(pd.crosstab(
    df_old_clean["class"],
    df_old_clean["split"]
))

In [ ]:
remaining_old = {
    normalize_path(p)
    for split in ["train", "val", "test"]
    for cls in CLASSES_3
    for p in (OLD_DIR / split / cls).iterdir()
    if p.is_file()
    and p.suffix.lower() in EXTENSIONS
    and normalize_path(p) not in quarantine_old
}

remaining_pairs = df_old_pairs[
    df_old_pairs["image_a"].map(normalize_path).isin(remaining_old)
    &
    df_old_pairs["image_b"].map(normalize_path).isin(remaining_old)
]

print("Cặp nghi trùng còn lại:", len(remaining_pairs))
print("Ảnh còn lại:", len(df_old_clean))
print("Số lớp:", df_old_clean["class"].nunique())

summary_old = pd.crosstab(
    df_old_clean["class"],
    df_old_clean["split"]
)

display(summary_old)

assert len(remaining_pairs) == 0
assert len(summary_old) == 6
assert {"train", "val", "test"}.issubset(summary_old.columns)
assert (summary_old[["train", "val", "test"]] > 0).all().all()

In [ ]:
import shutil
from pathlib import Path
from tqdm.auto import tqdm
import pandas as pd

DATASET_V2 = PROJECT_DIR / "datasets" / "efficientnet_v2"

SOURCES = [
    OLD_CLEAN_DIR,
    PROJECT_DIR / "datasets" / "efficientnet_v2_7_clean"
]

if DATASET_V2.exists() and any(
    p.is_file() for p in DATASET_V2.rglob("*")
):
    raise RuntimeError(
        "efficientnet_v2 đã có dữ liệu. "
        "Không ghi đè bản hiện tại."
    )

# Xác minh không có hai nguồn sử dụng cùng tên lớp
source_classes = []

for source in SOURCES:
    if not source.exists():
        raise FileNotFoundError(source)

    classes = {
        d.name
        for d in (source / "train").iterdir()
        if d.is_dir()
        and any(
            p.is_file() and p.suffix.lower() in EXTENSIONS
            for p in d.iterdir()
        )
    }

    source_classes.append(classes)

assert source_classes[0].isdisjoint(source_classes[1]), (
    "Hai bộ dữ liệu có lớp trùng tên, cần kiểm tra."
)

assert len(source_classes[0] | source_classes[1]) == 20

merged_records = []

for source_dir in SOURCES:
    for split in ["train", "val", "test"]:

        split_dir = source_dir / split

        for class_dir in sorted(split_dir.iterdir()):
            if not class_dir.is_dir():
                continue

            images = [
                p for p in class_dir.iterdir()
                if p.is_file()
                and p.suffix.lower() in EXTENSIONS
            ]

            for image_path in tqdm(
                images,
                desc=f"{source_dir.name}/{split}/{class_dir.name}"
            ):
                destination_dir = (
                    DATASET_V2 / split / class_dir.name
                )

                destination_dir.mkdir(
                    parents=True, exist_ok=True
                )

                destination = destination_dir / image_path.name

                if destination.exists():
                    raise RuntimeError(
                        f"Trùng tên file: {destination}"
                    )

                shutil.copy2(image_path, destination)

                merged_records.append({
                    "split": split,
                    "class": class_dir.name,
                    "source": str(image_path),
                    "destination": str(destination)
                })

df_merged = pd.DataFrame(merged_records)

df_merged.to_csv(
    REPORT_DIR / "efficientnet_v2_merged_manifest.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Tổng ảnh V2:", len(df_merged))
print("Số lớp:", df_merged["class"].nunique())

display(
    pd.crosstab(
        df_merged["class"],
        df_merged["split"]
    )
)

In [ ]:
import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor()
])

train_ds = datasets.ImageFolder(
    DATASET_V2 / "train",
    transform=transform
)

val_ds = datasets.ImageFolder(
    DATASET_V2 / "val",
    transform=transform
)

test_ds = datasets.ImageFolder(
    DATASET_V2 / "test",
    transform=transform
)

assert train_ds.class_to_idx == val_ds.class_to_idx
assert train_ds.class_to_idx == test_ds.class_to_idx

assert len(train_ds.classes) == 20, (
    f"Cần 20 lớp, hiện có {len(train_ds.classes)}"
)

print("Train:", len(train_ds))
print("Validation:", len(val_ds))
print("Test:", len(test_ds))
print("Tổng ảnh:", len(train_ds) + len(val_ds) + len(test_ds))
print("Số lớp:", len(train_ds.classes))

loader = DataLoader(
    train_ds,
    batch_size=8,
    shuffle=True,
    num_workers=0
)

images, labels = next(iter(loader))

print("Batch shape:", images.shape)
print("CUDA:", torch.cuda.is_available())

In [ ]:
from pathlib import Path
from collections import defaultdict
import hashlib

PROJECT_DIR = Path(r"C:\Users\ASUS\SIC-Food")
DATASET_V2 = PROJECT_DIR / "datasets" / "efficientnet_v2"

EXTENSIONS = {".jpg", ".jpeg", ".png", ".webp"}

hash_groups = defaultdict(list)

for split in ["train", "val", "test"]:
    for path in (DATASET_V2 / split).rglob("*"):
        if not path.is_file() or path.suffix.lower() not in EXTENSIONS:
            continue

        h = hashlib.sha256()

        with open(path, "rb") as f:
            for chunk in iter(lambda: f.read(1024 * 1024), b""):
                h.update(chunk)

        hash_groups[h.hexdigest()].append({
            "split": split,
            "path": str(path),
            "class": path.parent.name
        })

cross_split = []

for hash_value, items in hash_groups.items():
    splits = {item["split"] for item in items}

    if len(splits) > 1:
        cross_split.append({
            "sha256": hash_value,
            "files": items
        })

print("Tổng ảnh:", sum(len(x) for x in hash_groups.values()))
print("Nhóm trùng SHA256 giữa các split:", len(cross_split))

In [ ]:
import torch
import torch.nn as nn
from pathlib import Path
from torchvision import datasets
from torchvision.models import (
    efficientnet_b0,
    EfficientNet_B0_Weights
)
from torch.utils.data import DataLoader

PROJECT_DIR = Path(r"C:\Users\ASUS\SIC-Food")
DATASET_DIR = PROJECT_DIR / "datasets" / "efficientnet_v2"
MODEL_DIR = PROJECT_DIR / "models"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

weights = EfficientNet_B0_Weights.DEFAULT
preprocess = weights.transforms()

train_dataset = datasets.ImageFolder(
    DATASET_DIR / "train",
    transform=preprocess
)

val_dataset = datasets.ImageFolder(
    DATASET_DIR / "val",
    transform=preprocess
)

test_dataset = datasets.ImageFolder(
    DATASET_DIR / "test",
    transform=preprocess
)

assert train_dataset.class_to_idx == val_dataset.class_to_idx
assert train_dataset.class_to_idx == test_dataset.class_to_idx

NUM_CLASSES = len(train_dataset.classes)
assert NUM_CLASSES == 20

BATCH_SIZE = 16

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0
)

print("Device:", device)
print("Classes:", NUM_CLASSES)
print("Train:", len(train_dataset))
print("Validation:", len(val_dataset))
print("Test:", len(test_dataset))
print("Class mapping:", train_dataset.class_to_idx)

In [ ]:
model_v2 = efficientnet_b0(weights=weights)

for param in model_v2.features.parameters():
    param.requires_grad = False

in_features = model_v2.classifier[1].in_features

model_v2.classifier[1] = nn.Linear(
    in_features,
    NUM_CLASSES
)

model_v2 = model_v2.to(device)

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.AdamW(
    model_v2.classifier.parameters(),
    lr=0.001,
    weight_decay=1e-4
)

NUM_EPOCHS = 3

print("Model:", "EfficientNet-B0")
print("Output classes:", NUM_CLASSES)
print("Device:", device)
print("Epochs:", NUM_EPOCHS)

In [ ]:
import time
import copy
import pandas as pd
from tqdm.auto import tqdm

history_v2 = []
best_val_loss = float("inf")
best_model_state = None

CHECKPOINT_PATH = MODEL_DIR / "best_efficientnet_v2.pth"

for epoch in range(NUM_EPOCHS):
    start = time.time()

    # TRAIN
    model_v2.train()
    model_v2.features.eval()

    train_loss_sum = 0.0
    train_correct = 0
    train_total = 0

    for images, labels in tqdm(
        train_loader,
        desc=f"Epoch {epoch+1}/{NUM_EPOCHS} Train"
    ):
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad(set_to_none=True)

        outputs = model_v2(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        train_loss_sum += loss.item() * labels.size(0)
        train_correct += (
            outputs.argmax(1) == labels
        ).sum().item()
        train_total += labels.size(0)

    # VALIDATION
    model_v2.eval()

    val_loss_sum = 0.0
    val_correct = 0
    val_total = 0

    with torch.inference_mode():
        for images, labels in tqdm(
            val_loader,
            desc=f"Epoch {epoch+1}/{NUM_EPOCHS} Val"
        ):
            images = images.to(device)
            labels = labels.to(device)

            outputs = model_v2(images)
            loss = criterion(outputs, labels)

            val_loss_sum += loss.item() * labels.size(0)
            val_correct += (
                outputs.argmax(1) == labels
            ).sum().item()
            val_total += labels.size(0)

    train_loss = train_loss_sum / train_total
    train_acc = train_correct / train_total
    val_loss = val_loss_sum / val_total
    val_acc = val_correct / val_total

    history_v2.append({
        "epoch": epoch + 1,
        "train_loss": train_loss,
        "train_acc": train_acc,
        "val_loss": val_loss,
        "val_acc": val_acc
    })

    if val_loss < best_val_loss:
        best_val_loss = val_loss

        best_model_state = copy.deepcopy(
            model_v2.state_dict()
        )

        torch.save({
            "epoch": epoch + 1,
            "architecture": "efficientnet_b0",
            "model_state_dict": best_model_state,
            "optimizer_state_dict": optimizer.state_dict(),
            "class_to_idx": train_dataset.class_to_idx,
            "best_val_loss": best_val_loss
        }, CHECKPOINT_PATH)

        print("Đã lưu checkpoint V2 tốt nhất!")

    # Lưu lịch sử sau mỗi epoch
    pd.DataFrame(history_v2).to_csv(
        MODEL_DIR / "efficientnet_v2_history.csv",
        index=False,
        encoding="utf-8-sig"
    )

    elapsed = time.time() - start

    print(
        f"\nEpoch {epoch+1}/{NUM_EPOCHS}"
        f" | Train Loss: {train_loss:.4f}"
        f" | Train Acc: {train_acc:.4%}"
        f" | Val Loss: {val_loss:.4f}"
        f" | Val Acc: {val_acc:.4%}"
        f" | Time: {elapsed:.1f}s\n"
    )

if best_model_state is not None:
    model_v2.load_state_dict(best_model_state)

print("Hoàn thành huấn luyện thử nghiệm!")
print("Best Val Loss:", best_val_loss)
print("Checkpoint:", CHECKPOINT_PATH)

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    ConfusionMatrixDisplay
)
import matplotlib.pyplot as plt

model_v2.eval()

y_true = []
y_pred = []

with torch.inference_mode():
    for images, labels in tqdm(val_loader, desc="Evaluate V2"):
        images = images.to(device)

        outputs = model_v2(images)
        predictions = outputs.argmax(dim=1)

        y_true.extend(labels.tolist())
        y_pred.extend(predictions.cpu().tolist())

accuracy = accuracy_score(y_true, y_pred)
macro_f1 = f1_score(
    y_true, y_pred,
    average="macro",
    zero_division=0
)

print(f"Validation Accuracy: {accuracy:.4%}")
print(f"Validation Macro-F1: {macro_f1:.4f}")

print(classification_report(
    y_true,
    y_pred,
    labels=list(range(NUM_CLASSES)),
    target_names=val_dataset.classes,
    zero_division=0
))

fig, ax = plt.subplots(figsize=(15, 13))

ConfusionMatrixDisplay.from_predictions(
    y_true,
    y_pred,
    labels=list(range(NUM_CLASSES)),
    display_labels=val_dataset.classes,
    xticks_rotation=90,
    cmap="Blues",
    ax=ax
)

plt.tight_layout()
plt.show()

In [ ]:
history_df = pd.read_csv(
    MODEL_DIR / "efficientnet_v2_history.csv"
)

display(history_df)

for metric in ["loss", "acc"]:
    plt.figure(figsize=(8, 5))

    plt.plot(
        history_df["epoch"],
        history_df[f"train_{metric}"],
        marker="o",
        label=f"Train {metric}"
    )

    plt.plot(
        history_df["epoch"],
        history_df[f"val_{metric}"],
        marker="o",
        label=f"Validation {metric}"
    )

    plt.xlabel("Epoch")
    plt.ylabel(metric.upper())
    plt.title(f"EfficientNet V2 - {metric.upper()}")
    plt.legend()
    plt.grid(alpha=0.3)
    plt.show()

In [ ]:
from pathlib import Path
from collections import Counter
import pandas as pd

PROJECT_DIR = Path(r"C:\Users\ASUS\SIC-Food")

DATASET_DIRS = {
    "EfficientNet V3": PROJECT_DIR / "datasets/efficientnet_v3",
    "Cabbage V4": PROJECT_DIR / "datasets/efficientnet_v4_cabbage_candidates/train",
    "V4 Current": PROJECT_DIR / "datasets/state_v4_current_candidates",
}

EXTS = {".jpg", ".jpeg", ".png", ".webp"}

records = []

for dataset_name, root in DATASET_DIRS.items():
    if not root.exists():
        print("KHÔNG TỒN TẠI:", root)
        continue

    for class_dir in root.rglob("*"):
        if not class_dir.is_dir():
            continue

        class_name = class_dir.name

        if "__" not in class_name:
            continue

        image_count = sum(
            1 for p in class_dir.iterdir()
            if p.is_file() and p.suffix.lower() in EXTS
        )

        if image_count == 0:
            continue

        ingredient, state = class_name.split("__", 1)

        records.append({
            "dataset": dataset_name,
            "ingredient": ingredient,
            "state": state,
            "class": class_name,
            "images": image_count,
            "path": str(class_dir)
        })

df_inventory = pd.DataFrame(records)

print("===== DANH SÁCH DATASET =====")
display(df_inventory)

if not df_inventory.empty:
    print("\n===== TỔNG HỢP THEO NGUYÊN LIỆU =====")

    summary = df_inventory.pivot_table(
        index="ingredient",
        columns="state",
        values="images",
        aggfunc="sum",
        fill_value=0
    )

    summary["total_images"] = summary.sum(axis=1)

    summary["two_states_available"] = (
        (summary.get("normal_appearance", 0) > 0)
        & (summary.get("visible_defect", 0) > 0)
    )

    display(summary)

    print("\nTổng nguyên liệu:", len(summary))
    print("Đủ hai trạng thái:",
          int(summary["two_states_available"].sum()))

In [ ]:
from PIL import Image
from IPython.display import display
import random

for ingredient in sorted(df_inventory["ingredient"].unique()):

    print("\n" + "=" * 55)
    print("NGUYÊN LIỆU:", ingredient.upper())

    part = df_inventory[
        df_inventory["ingredient"] == ingredient
    ]

    for _, row in part.iterrows():

        print(
            "\nTrạng thái:",
            row["state"],
            "| Dataset:",
            row["dataset"]
        )

        folder = Path(row["path"])

        images = [
            p for p in folder.iterdir()
            if p.suffix.lower() in EXTS
        ]

        for image_path in random.Random(42).sample(
            images, min(2, len(images))
        ):
            with Image.open(image_path) as im:
                preview = im.convert("RGB")
                preview.thumbnail((220, 220))
                display(preview.copy())